# G05 — Frozen thesis-style WAS-SSL: full ISIC 2019 and HAM/BCN source subsets

Replays the original thesis and R07 frozen ensemble artifacts. No retraining, no new weights, no per-source optimization.

**Selected formula (frozen from the exploratory G04 comparison):** `P=0.60*P_weighted+0.40*P_logistic_stacker`, **without class bias**.

Image-aware full original J1 test and lesion-aware full N01 validation are reported separately; the N01 locked test is **never opened**.

Attach four Kaggle outputs: `last-part-of-the-previous-one`, `confidence-gated-ensemble`, `lesion-r07-n01-all-ensembles-s42`, `g03a-image-level-all-ensembles-s42`, `g03b-lesion-aware-all-ensembles-s42` (five outputs). Use CPU, Internet OFF.

**G05 v2:** The original Rotation CSV stores probability columns as `prob_0_<class>`, while fusion CSVs store `prob_class_0`. Both are now parsed in numeric class-index order and validated against the frozen eight-class protocol. No prediction arrays, weights, stacker parameters, or model selection logic are changed.

## 1. Imports, metrics, discovery, and read-only validation

In [1]:
import os,json,hashlib,time,re,re
from pathlib import Path
from collections import OrderedDict
import numpy as np,pandas as pd,joblib
from scipy.special import softmax
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,roc_auc_score,average_precision_score,matthews_corrcoef,log_loss,confusion_matrix

SEED=42;CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path(os.getenv('G05_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')).resolve()
WORK=Path(os.getenv('G05_WORK_DIR','/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')).resolve()
OUT=WORK/'g05-frozen-was-full-isic2019-s42'
for d in ['tables','reports','predictions']:(OUT/d).mkdir(parents=True,exist_ok=True)
ALPHA=.4;N_BOOT=int(os.getenv('G05_BOOTSTRAP','1000'))
MODEL_NAMES=['SimCLR_NoTTA','SimCLR_SelectedTTA','Rotation','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']


def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def write(path,object):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(object,indent=2,default=str,allow_nan=False));tmp.replace(path)
def read(path):return json.loads(Path(path).read_text())
def one(paths,description):
    paths=sorted(set(Path(p).resolve() for p in paths if Path(p).is_file()))
    req(len(paths)==1,f'{description}: expected exactly one artifact, found {len(paths)}: {[str(p) for p in paths[:7]]}')
    return paths[0]
def prob(a,n=None):
    a=np.asarray(a,dtype=np.float64)
    req(a.ndim==2 and a.shape[1]==8 and (n is None or len(a)==n),'Invalid prediction dimensions')
    req(np.isfinite(a).all() and (a>=0).all() and np.max(abs(a.sum(1)-1))<.002,'Invalid prediction probabilities')
    return np.maximum(a,1e-12)/np.maximum(a,1e-12).sum(1,keepdims=True)
def mix(a,b,alpha=ALPHA):return prob((1-alpha)*a+alpha*b)
def tscale(p,t):
    req(np.isfinite(t) and t>0,'Invalid fitted temperature')
    return prob(softmax(np.log(np.clip(p,1e-12,1))/t,axis=1))
def bias(p,b):
    b=np.asarray(b,dtype=np.float64);req(b.shape==(8,) and np.isfinite(b).all(),'Invalid class-bias vector')
    return prob(softmax(np.log(np.clip(p,1e-12,1))+b,axis=1))
def arrstr(z,k):
    a=z[k];req(a.ndim==1 and a.dtype.kind in 'OUS',f'Invalid text metadata: {k}')
    if a.dtype.kind=='O':req(all(isinstance(x,(str,np.str_)) for x in a),f'Unexpected object values: {k}')
    return a.astype(str)
def metric(y,p):
    y=np.asarray(y,dtype=int);p=prob(p,len(y));pred=p.argmax(1);observed=np.unique(y)
    req(len(y)>0 and np.isin(y,np.arange(8)).all(),'Missing/invalid ground-truth labels')
    auc=[];pr=[]
    for c in observed:
        pos=(y==c).astype(int)
        if pos.min()==pos.max():continue
        auc.append(float(roc_auc_score(pos,p[:,c])))
        pr.append(float(average_precision_score(pos,p[:,c])))
    ece=0.;conf=p.max(1)
    for j in range(15):
        m=(conf>=j/15)&(conf<(j+1)/15 if j<14 else conf<=1)
        if m.any():ece+=m.mean()*abs(np.mean(pred[m]==y[m])-np.mean(conf[m]))
    return {'n':int(len(y)),'present_classes':int(len(observed)),
        'accuracy':float(accuracy_score(y,pred)),'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
        'macro_f1_present_classes':float(f1_score(y,pred,labels=observed,average='macro',zero_division=0)),
        'macro_f1_all8':float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),
        'roc_auc_macro_present':float(np.mean(auc)) if auc else None,
        'pr_auc_macro_present':float(np.mean(pr)) if pr else None,
        'mcc':float(matthews_corrcoef(y,pred)),
        'brier_8class':float(np.mean(np.sum((p-np.eye(8)[y])**2,axis=1))),
        'nll_8class':float(log_loss(y,p,labels=np.arange(8))),
        'ece15':float(ece),'unsupported_prediction_rate':float(np.mean(~np.isin(pred,observed)))}

def bacc_ci(y,p,groups=None,B=N_BOOT,seed=SEED):
    y=np.asarray(y,dtype=int);pred=np.asarray(p).argmax(1);classes=np.unique(y)
    if B==0:return (None,None,'DISABLED')
    rng=np.random.default_rng(seed)
    if groups is not None:
        g=np.asarray(groups,dtype=str)
        valid=(len(g)==len(y) and np.all(g!='') and np.all(g!='UNKNOWN') and len(set(g))<len(g))
    else:valid=False
    vals=[]
    if valid:
        unique,inv=np.unique(g,return_inverse=True)
        buckets=[np.flatnonzero(inv==j) for j in range(len(unique))]
        for _ in range(B):
            ids=np.concatenate([buckets[j] for j in rng.integers(len(unique),size=len(unique))])
            if len(np.unique(y[ids]))!=len(classes):continue
            vals.append(np.mean([(pred[ids][y[ids]==c]==c).mean() for c in classes]))
        method='LESION_CLUSTER_BOOTSTRAP'
    else:
        buckets=[np.flatnonzero(y==c) for c in classes]
        for _ in range(B):
            ix=np.concatenate([rng.choice(b,size=len(b),replace=True) for b in buckets])
            vals.append(np.mean([(pred[ix][y[ix]==c]==c).mean() for c in classes]))
        method='CLASS_STRATIFIED_IMAGE_BOOTSTRAP'
    if len(vals)<max(100,B//5):return (None,None,'INSUFFICIENT_VALID_GROUP_REPLICATES')
    lo,hi=np.percentile(vals,[2.5,97.5])
    return float(lo),float(hi),method

def labelcol(df):
    for k in ['y_true','true_label','label']:
        if k in df:return k
    raise RuntimeError('Saved prediction CSV missing label column')
def csvprob(path):
    f=pd.read_csv(path)
    formats=[r'prob_class_([0-7])',r'prob_([0-7])(?:_(.+))?']
    cols=None;matched_format=None
    for fmt in formats:
        indexed={}
        for c in f.columns:
            m=re.fullmatch(fmt,str(c))
            if not m:continue
            i=int(m.group(1))
            req(i not in indexed,f'{path}: duplicate probability class index {i}')
            if len(m.groups())>1 and m.group(2):
                suffix=m.group(2).strip().upper()
                if suffix in CLASSES:req(suffix==CLASSES[i],f'{path}: class name/order conflict in {c}')
                elif suffix.isdigit():req(int(suffix)==i,f'{path}: class index/order conflict in {c}')
            indexed[i]=c
        if indexed:
            req(set(indexed)==set(range(8)),f'{path}: incomplete probability columns, found={indexed}')
            cols=[indexed[i] for i in range(8)];matched_format=fmt;break
    req(cols is not None,f'{path}: no supported probability columns; available={f.columns.tolist()}')
    y=pd.to_numeric(f[labelcol(f)],errors='raise').to_numpy(dtype=np.int64)
    req(np.isin(y,np.arange(8)).all(),f'{path}: labels must be integer class IDs 0..7')
    p=prob(f[cols].to_numpy(dtype=np.float64))
    req(len(y)==len(p),f'{path}: label/probability count mismatch')
    return y,p

def resolve_thesis():
    found=[]
    for p in INPUT.rglob('final_ensemble_config.json'):
        try:
            j=read(p)
            if j.get('models')==MODEL_NAMES and all(k in j for k in ['temperatures','optimized_weights','stacking_alpha','class_bias']):
                n=p.parent/'final_internal_probabilities.npz';m=p.parent/'stacking_meta_model.joblib'
                if n.is_file() and m.is_file():found.append((p,n,m,j))
        except Exception:pass
    req(len(found)==1,'Attach the original `confidence-gated-ensemble` output with final_ensemble_config.json, final_internal_probabilities.npz, and stacking_meta_model.joblib (exactly one copy).')
    return found[0]


## 2. Original thesis: reconstruct seven-source weighted and fitted logistic stacker; verify saved full-test probabilities

In [2]:

config_file,original_npz,stack_file,cfg=resolve_thesis()
print('Thesis fitted artifact:',config_file)
markers=[p for p in INPUT.rglob('best_weights.json') if p.parent.name=='simclr_dominant_weighted_ensemble_v2']
marker=one(markers,'Original thesis image-level source: output of last-part-of-the-previous-one, containing simclr_dominant_weighted_ensemble_v2/best_weights.json')
PIPE=marker.parent.parent
source_paths=OrderedDict([
 ('SimCLR_NoTTA',('npz',PIPE/'simclr_finetune_multiseed_bacc_v2/multiseed_probabilities_no_tta.npz')),
 ('SimCLR_SelectedTTA',('npz',PIPE/'simclr_finetune_multiseed_bacc_v2/tta_selection/selected_multiseed_tta_probabilities.npz')),
 ('Rotation',('csv',PIPE/'extended_imagenet_rotation_finetune/imagenet_rotation_val_predictions.csv',PIPE/'extended_imagenet_rotation_finetune/imagenet_rotation_test_predictions.csv')),
 ('FusionConcat',('csv',PIPE/'fusion_concat_multiseed_rep_v2/full_evaluation/val/val_predictions.csv',PIPE/'fusion_concat_multiseed_rep_v2/full_evaluation/test/test_predictions.csv')),
 ('FusionGatedV2',('csv',PIPE/'fusion_gated_v2_multiseed_rep_v2/full_evaluation/val/val_predictions.csv',PIPE/'fusion_gated_v2_multiseed_rep_v2/full_evaluation/test/test_predictions.csv')),
 ('FusionGatedV3',('csv',PIPE/'fusion_gated_v3_multiseed_rep_v2/full_evaluation/val/val_predictions.csv',PIPE/'fusion_gated_v3_multiseed_rep_v2/full_evaluation/test/test_predictions.csv')),
 ('ResidualFusion',('npz',PIPE/'residual_simclr_rotation_fusion_v1/multiseed_probabilities.npz'))])
raw={};y_test=y_val=None
for name,spec in source_paths.items():
    req(all(p.is_file() for p in spec[1:]),f'Original trained source missing for {name}: {spec[1:]}')
    if spec[0]=='npz':
        with np.load(spec[1],allow_pickle=False) as z:
            yv=z['y_val'].astype(int);yt=z['y_test'].astype(int);p=prob(z['test_prob'])
    else:
        yv,_=csvprob(spec[1]);yt,p=csvprob(spec[2])
    if y_val is None:y_val=yv;y_test=yt
    else:req(np.array_equal(yv,y_val) and np.array_equal(yt,y_test),f'Original J1 row order/labels differ: {name}')
    raw[name]=p
req(len(y_test)==2534 and len(y_val)==2533,'Original J1 split sizes differ from locked protocol')
req(set(cfg['temperatures'])==set(MODEL_NAMES) and set(cfg['optimized_weights'])==set(MODEL_NAMES),'Original thesis configuration sources mismatch')
cal=OrderedDict((k,tscale(raw[k],float(cfg['temperatures'][k]))) for k in MODEL_NAMES)
w=np.asarray([float(cfg['optimized_weights'][k]) for k in MODEL_NAMES]);req(np.all(w>=0) and abs(w.sum()-1)<.0003,'Thesis weighted coefficients invalid')
weighted=prob(sum(w[i]*cal[k] for i,k in enumerate(MODEL_NAMES)))

def old_features(p):
    blocks=[]
    for k in MODEL_NAMES:
        a=p[k]
        blocks.extend([a,np.log(np.clip(a,1e-12,1)),np.max(a,axis=1,keepdims=True),
                       -np.sum(a*np.log(np.clip(a,1e-12,1)),axis=1,keepdims=True)])
    return np.concatenate(blocks,axis=1).astype(np.float32)

# A joblib file contains Python pickle data. Only use the user's own trusted Kaggle output.
stacker=joblib.load(stack_file)
stack=stacker.predict_proba(old_features(cal))
clf=stacker.named_steps.get('classifier') if hasattr(stacker,'named_steps') else stacker
stack_classes=np.asarray(clf.classes_,dtype=int)
req(set(stack_classes)==set(range(8)),'Historical stacker missing classes')
stack=prob(stack[:,np.argsort(stack_classes)])
a_original=float(cfg['stacking_alpha']);req(0<a_original<1,'Original stacker alpha invalid')
legacy_blend=mix(cal['SimCLR_NoTTA'],stack,a_original)
legacy_bias=bias(legacy_blend,cfg['class_bias'])
with np.load(original_npz,allow_pickle=False) as z:
    req(np.array_equal(z['y_test'].astype(int),y_test),'Original J1 final NPZ labels disagree with base probability sources')
    audits={
        'optimized_weighted_max_abs':float(np.max(abs(prob(z['weighted_test_prob'])-weighted))),
        'original_blend_max_abs':float(np.max(abs(prob(z['stacking_test_prob'])-legacy_blend))),
        'original_bias_max_abs':float(np.max(abs(prob(z['stacking_bias_test_prob'])-legacy_bias)))}
    req(max(audits.values())<.002,f'Original thesis reconstruction DOES NOT match archived full-test results: {audits}')
WAS_A=mix(weighted,stack,ALPHA)
ORIG_FULL=OrderedDict([('OptimizedWeighted',weighted),('LogisticStacking',stack),
    ('OriginalSimCLRAnchor_NoBias',legacy_blend),('OriginalCalStack_ClassBias',legacy_bias),
    ('WeightedAnchor_NoBias_alpha0.4',WAS_A),('WeightedAnchor_OriginalClassBias',bias(WAS_A,cfg['class_bias']))])
write(OUT/'reports/original_internal_replay_audit.json',{'source_root':str(PIPE),'config_sha256':sha(config_file),
    'stacker_sha256':sha(stack_file),'full_test_npz_sha256':sha(original_npz),**audits})
print('Original J1 full-test reconstruction: PASSED',audits)


Thesis fitted artifact: /kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_ensemble_config.json
Original J1 full-test reconstruction: PASSED {'optimized_weighted_max_abs': 6.929296414437403e-08, 'original_blend_max_abs': 9.074280066245777e-08, 'original_bias_max_abs': 1.1006265515645097e-07}


## 3. N01: load R07 frozen six-source candidates for 2,186 full-validation images

In [3]:

# Load complete frozen N01 validation predictions. This stage NEVER reads N01 locked test.
r07=[]
for p in INPUT.rglob('final_ensemble_config.json'):
    try:
        j=read(p)
        if j.get('stage')=='R07' and j.get('status')=='COMPLETED':
            if (p.parent.parent/'predictions/final_validation.npz').is_file():r07.append((p,j))
    except Exception:pass
req(len(r07)==1,'Attach exactly one completed lesion-r07-n01-all-ensembles-s42 output with reports/final_ensemble_config.json, predictions/final_validation.npz and candidate_*.npy')
r07_file,r07_config=r07[0];r07_root=r07_file.parent.parent
req(r07_config.get('test_touched') is False,'R07 claims to have touched the N01 locked test')
with np.load(r07_root/'predictions/final_validation.npz',allow_pickle=True) as z:
    y_n01=np.asarray(z['y'],dtype=int);ids_n01=arrstr(z,'image_id');groups_n01=arrstr(z,'group_id')
    if 'classes' in z:req(arrstr(z,'classes').tolist()==CLASSES,'R07 class order mismatch')
req(len(y_n01)==len(ids_n01)==2186 and len(set(ids_n01))==2186,'R07 full validation incomplete/nonunique')

B_WEIGHT='Weighted_AllConstrained_1';B_STACK='Stack_extended_balanced'
B_WAS=f'Blend_{B_STACK}__{B_WEIGHT}__a0.4'
B_SIM=f'Blend_{B_STACK}__Cal_simclr__a0.6'
B_LEGACY='ClassBias_'+B_WAS
all_specs=r07_config['all_candidate_specs']
for name in [B_WEIGHT,B_STACK,B_WAS,B_SIM,B_LEGACY]:
    req(name in all_specs,f'R07 missing frozen recipe for {name}')

def r07proba(name):
    p=r07_root/'predictions'/f'candidate_{name}.npy'
    req(p.is_file(),f'Missing R07 validation prediction: {p.name}')
    return prob(np.load(p,allow_pickle=False),len(y_n01))

N_WEIGHT=r07proba(B_WEIGHT);N_STACK=r07proba(B_STACK);N_WAS=r07proba(B_WAS)
req(all_specs[B_WAS].get('kind')=='blend' and all_specs[B_WAS].get('anchor')==B_WEIGHT and
    all_specs[B_WAS].get('stack')==B_STACK and np.isclose(all_specs[B_WAS]['alpha'],ALPHA),
    'N01 weighted-anchor recipe mismatch')
req(np.max(abs(N_WAS-mix(N_WEIGHT,N_STACK)))<.002,'R07 weighted-anchor WAS replay mismatch')
N_BIAS=r07proba(B_LEGACY)
req(all_specs[B_LEGACY].get('kind')=='class_bias' and all_specs[B_LEGACY].get('parent')==B_WAS,
    'R07 class-bias candidate has a different parent')
req(np.max(abs(N_BIAS-bias(N_WAS,all_specs[B_LEGACY]['offsets'])))<.002,
    'R07 frozen bias replay mismatch')
N_FULL=OrderedDict([('OptimizedWeighted',N_WEIGHT),('LogisticStacking',N_STACK),
 ('OriginalSimCLRAnchor_NoBias',r07proba(B_SIM)),('OriginalCalStack_ClassBias',N_BIAS),
 ('WeightedAnchor_NoBias_alpha0.4',N_WAS),
 ('WeightedAnchor_OriginalClassBias',bias(N_WAS,cfg['class_bias']))])


## 4. Freeze WAS-SSL, then evaluate full ISIC 2019 J1 test and N01 validation

In [4]:

# Record frozen candidate BEFORE computing/reporting any performance.
recipe={'name':'WeightedAnchor_NoBias_alpha0.4','status':'FROZEN_EXPLORATORY_CANDIDATE',
        'selection_basis':'Prior G04 source-stratified exploratory family mean, not new ISIC test',
        'formula':'P=0.60*P_optimized_weighted+0.40*P_fitted_logistic_stacking',
        'anchor_weight':.60,'stack_weight':.40,'class_bias':None,
        'original_j1':{'weights':dict(cfg['optimized_weights']),'temperatures':dict(cfg['temperatures']),
                       'stacker_sha256':sha(stack_file),'config_sha256':sha(config_file)},
        'lesion_n01':{'weights_candidate':B_WEIGHT,'stack_candidate':B_STACK,'blend_candidate':B_WAS,
                      'frozen_r07_sha256':sha(r07_file),'n01_manifest_sha256':r07_config['n01_manifest_sha256']},
        'N01_test_touched':False,'protocol_warning':'G04 comparisons and original J1 test previously examined; cannot claim independent unbiased selection.'}
lock=OUT/'reports/frozen_was_recipe.json'
if lock.is_file():
    prev=read(lock);req(prev==recipe,'A different frozen recipe already exists in this output; refusing to overwrite')
else:write(lock,recipe)

# Complete ISIC 2019 benchmark. N01 val is not a test; J1 full test is already historically examined.
FULL=[];RECALL=[];CI=[]
for protocol,y,imgs,grp in [('image_aware_J1_full_test',y_test,ORIG_FULL,None),
                            ('lesion_aware_N01_full_validation',y_n01,N_FULL,groups_n01)]:
    for family,p in imgs.items():
        m=metric(y,p);lo,hi,method=bacc_ci(y,p,grp,seed=SEED+len(FULL))
        FULL.append({'protocol':protocol,'family':family,**m,'bacc_ci_lower':lo,'bacc_ci_upper':hi,'ci_type':method})
        pr=p.argmax(1)
        for c in np.unique(y):
            mask=y==c;RECALL.append({'protocol':protocol,'family':family,'class':CLASSES[c],
                    'n':int(mask.sum()),'recall':float(np.mean(pr[mask]==c))})
        cm=pd.DataFrame(confusion_matrix(y,pr,labels=np.arange(8)),index=CLASSES,columns=CLASSES)
        cm.to_csv(OUT/'tables'/f'confusion_{protocol}_{family}.csv')
full=pd.DataFrame(FULL);full.to_csv(OUT/'tables/full_isic2019_comparison.csv',index=False)
pd.DataFrame(RECALL).to_csv(OUT/'tables/full_isic2019_per_class_recall.csv',index=False)


## 5. Evaluate the same frozen families on HAM10000/BCN20000 source subsets; export results

In [5]:

# Replay the same frozen model family on four source-subsets using G03 archives.
def load_g03(stage):
    found=[]
    for p in INPUT.rglob('final_status.json'):
        try:
            j=read(p)
            if j.get('stage')==stage and j.get('status')=='COMPLETED':
                root=p.parent.parent
                arc=root/'predictions/ensemble_probabilities.npz'
                if arc.is_file():found.append((p,arc,j))
        except Exception:pass
    req(len(found)==1,f'Attach exactly one {stage} complete output with predictions/ensemble_probabilities.npz')
    status_path,archive,status=found[0]
    with np.load(archive,allow_pickle=True) as z:
        names=arrstr(z,'candidate_names');ids=arrstr(z,'image_id');source=arrstr(z,'source');
        ys=np.asarray(z['y'],dtype=int);p=np.asarray(z['proba'],dtype=np.float64)
        req(arrstr(z,'classes').tolist()==CLASSES,f'{stage}: incompatible class order')
        req(p.shape==(len(names),len(ids),8) and len(set(ids))==len(ids),f'{stage}: invalid prediction dimensions')
        req(len(names)==int(status['candidate_count']) and len(ids)==int(status['n_images']),f'{stage}: status count mismatch')
        ix={n:i for i,n in enumerate(names)}
    return {'ids':ids,'source':source,'y':ys,'p':p,'lookup':ix,'sha':sha(archive),'status':status}

a03=load_g03('G03-A');b03=load_g03('G03-B')
req(a03['status']['historical_replay']['available'],'G03-A requires original fitted thesis stacker')
req(b03['status'].get('source_reproduction_quality')=='CLOSE' and b03['status'].get('test_touched') is False,'G03-B provenance mismatch')

def getp(d,key):
    req(key in d['lookup'],f'Missing candidate {key} in G03')
    return prob(d['p'][d['lookup'][key]])

SRC=[]
for protocol,d,keys in [('image_aware',a03,{
    'OptimizedWeighted':'OriginalThesis_ConstrainedWeighted',
    'LogisticStacking':'OriginalThesis_StackOnly',
    'OriginalSimCLRAnchor_NoBias':'OriginalThesis_StackBlend_NoBias',
    'OriginalCalStack_ClassBias':'OriginalThesis_CalStack_ClassBias'}),
    ('lesion_aware',b03,{
    'OptimizedWeighted':B_WEIGHT,'LogisticStacking':B_STACK,
    'OriginalSimCLRAnchor_NoBias':B_SIM,'OriginalCalStack_ClassBias':B_LEGACY})]:
    pred={k:getp(d,v) for k,v in keys.items()}
    pred['WeightedAnchor_NoBias_alpha0.4']=mix(pred['OptimizedWeighted'],pred['LogisticStacking'])
    pred['WeightedAnchor_OriginalClassBias']=bias(pred['WeightedAnchor_NoBias_alpha0.4'],cfg['class_bias'])
    source=d['source'];y=d['y']
    req(set(source)=={'HAM10000','BCN20000'},f'{protocol}: unexpected source labels')
    for ds in ['HAM10000','BCN20000']:
        ix=np.flatnonzero(source==ds)
        for fam,pp in pred.items():
            SRC.append({'protocol':protocol,'cohort':ds,'family':fam,**metric(y[ix],pp[ix])})
source_df=pd.DataFrame(SRC)
source_df.to_csv(OUT/'tables/ham_bcn_thesis_family_comparison.csv',index=False)

# The four-source average is descriptive; it has already been inspected in G04.
order=['image_aware_HAM10000','image_aware_BCN20000','lesion_aware_HAM10000','lesion_aware_BCN20000']
rank=source_df.pivot_table(index='family',columns=['protocol','cohort'],values='balanced_accuracy')
rank.columns=[a+'_'+b for a,b in rank.columns]
req(set(rank.columns)==set(order),'Four-source comparison incomplete')
rank=rank[order];rank['mean_bacc_four_cohorts']=rank[order].mean(axis=1)
rank['worst_cohort_bacc']=rank[order].min(axis=1)
rank=rank.sort_values('mean_bacc_four_cohorts',ascending=False)
rank.to_csv(OUT/'tables/frozen_family_four_source_ranking.csv')

# No optimization and no re-selection from these FULL J1 test and N01 validation results.
result={'stage':'G05','status':'COMPLETED','candidate':'WeightedAnchor_NoBias_alpha0.4',
        'frozen_recipe_sha256':sha(lock),'full_j1_test_n':int(len(y_test)),
        'full_n01_validation_n':int(len(y_n01)),
        'g03_a_sha256':a03['sha'],'g03_b_sha256':b03['sha'],
        'r07_config_sha256':sha(r07_file),'original_config_sha256':sha(config_file),
        'test_touched_N01':False,'selection_performed_in_G05':False,
        'source_only_not_external_validation':True,
        'limitations':'Original J1 test and N01 validation have been examined earlier; final independent lesion-aware locked test awaits a separate notebook.'}
write(OUT/'reports/final_status.json',result)
print('\nFULL ISIC 2019 BACC (%) — frozen methods')
print((full.pivot(index='family',columns='protocol',values='balanced_accuracy')*100).round(2).to_string())
print('\nHAM / BCN SOURCE SUBSETS BACC (%) — same frozen methods')
print((rank*100).round(2).to_string())
print('\nStatus:',OUT/'reports/final_status.json')



FULL ISIC 2019 BACC (%) — frozen methods
protocol                          image_aware_J1_full_test  lesion_aware_N01_full_validation
family                                                                                      
LogisticStacking                                     82.99                             58.11
OptimizedWeighted                                    81.03                             60.69
OriginalCalStack_ClassBias                           83.95                             62.04
OriginalSimCLRAnchor_NoBias                          84.00                             59.21
WeightedAnchor_NoBias_alpha0.4                       83.29                             61.69
WeightedAnchor_OriginalClassBias                     83.43                             59.95

HAM / BCN SOURCE SUBSETS BACC (%) — same frozen methods
                                  image_aware_HAM10000  image_aware_BCN20000  lesion_aware_HAM10000  lesion_aware_BCN20000  mean_bacc_four_cohorts  worst_coh